# Pipeline NLP : comédie ou horreur

## Partie 1 — Audit et nettoyage

Objectifs de cette partie :

- charger le CSV source sans le modifier ;
- vérifier son schéma, les valeurs manquantes et les classes ;
- mesurer la longueur des résumés ;
- repérer les doublons exacts et les éventuels conflits d'étiquettes ;
- créer en mémoire un jeu nettoyé avant tout découpage.

Convention : `0 = comédie` et `1 = horreur`.

### 1.1 Imports et chemin du fichier

`Path` construit des chemins compatibles avec Windows. Le bloc ci-dessous fonctionne lorsque le notebook est lancé depuis la racine du projet ou depuis le dossier `notebooks`. Une assertion arrête immédiatement l'exécution si le CSV est introuvable.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "textes_films_horreur_comedie.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "textes_films_horreur_comedie.csv"
assert DATA_PATH.exists(), f"CSV introuvable : {DATA_PATH}"

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Fichier source    : {DATA_PATH}")

Racine du projet : C:\Users\driou\OneDrive\Bureau\modeleIA
Fichier source    : C:\Users\driou\OneDrive\Bureau\modeleIA\textes_films_horreur_comedie.csv


### 1.2 Chargement et schéma

Une ligne de `DataFrame` représente un film. Les colonnes attendues sont le titre, le résumé et la cible. Nous recalculons les dimensions à chaque exécution au lieu de les saisir comme résultats fixes.

In [2]:
df_raw = pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS = ["titre", "resume", "cible"]

print(f"Dimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
display(df_raw.head(3))
display(df_raw.dtypes.rename("type pandas").to_frame())

assert list(df_raw.columns) == EXPECTED_COLUMNS, (
    f"Colonnes inattendues : {list(df_raw.columns)}"
)

Dimensions : 2400 lignes × 3 colonnes


,titre,resume,cible
0,Chiedo asilo,"Roberto has a new job as the teacher at a nursery school. The first child he meets is Gianluigi, who's mute. Roberto...",0
1,Friday the 13th: A New Beginning,"Five years after killing the goalie hockey-masked killer Jason Voorhees, Tommy Jarvis has grown up in various mental...",1
2,Stranger Than Fiction,Everybody knows that your life is a story. But what if a story was your life? Harold Crick is your average IRS agent...,0


,type pandas
titre,str
resume,str
cible,int64


### 1.3 Valeurs manquantes, textes vides et classes

Une valeur manquante est représentée par `NaN`. Un texte vide peut ne pas être manquant : par exemple, une chaîne ne contenant que des espaces. Nous contrôlons donc séparément ces deux cas.

In [3]:
missing_counts = df_raw.isna().sum()
empty_counts = pd.Series(
    {
        column: df_raw[column].fillna("").astype(str).str.strip().eq("").sum()
        for column in ["titre", "resume"]
    },
    name="textes vides",
)

display(missing_counts.rename("valeurs manquantes").to_frame())
display(empty_counts.to_frame())

class_counts = df_raw["cible"].value_counts().sort_index()
class_proportions = df_raw["cible"].value_counts(normalize=True).sort_index()
class_summary = pd.DataFrame(
    {"effectif": class_counts, "proportion": class_proportions}
)
display(class_summary)

,valeurs manquantes
titre,0
resume,0
cible,0


,textes vides
titre,0
resume,0


,effectif,proportion
cible,,
0,1200,0.5
1,1200,0.5


### 1.4 Longueur des résumés

La longueur en caractères donne une indication de stockage. La longueur en mots aide à comprendre la quantité de texte disponible par observation. `describe()` fournit notamment le minimum, la médiane (`50%`) et le maximum.

In [4]:
resume_lengths = pd.DataFrame(
    {
        "caracteres": df_raw["resume"].str.len(),
        "mots": df_raw["resume"].str.split().str.len(),
    }
)
display(resume_lengths.describe().round(2))

,caracteres,mots
count,2400.00,2400.00
mean,558.19,96.70
std,349.28,61.26
min,201.00,30.00
25%,335.00,59.00
50%,474.00,82.00
75%,679.00,117.00
max,7104.00,1287.00


### 1.5 Doublons exacts

`duplicated(keep=False)` sélectionne toutes les lignes appartenant à un groupe répété. `duplicated(keep="first")` compte seulement les occurrences excédentaires à supprimer : la première occurrence de chaque groupe est conservée.

In [5]:
exact_duplicate_mask = df_raw.duplicated(
    subset=EXPECTED_COLUMNS, keep=False
)
duplicate_excess_count = int(
    df_raw.duplicated(subset=EXPECTED_COLUMNS, keep="first").sum()
)
duplicate_groups = (
    df_raw.loc[exact_duplicate_mask]
    .value_counts(subset=EXPECTED_COLUMNS)
    .rename("taille_groupe")
    .reset_index()
)

print(f"Lignes appartenant à un groupe répété : {exact_duplicate_mask.sum()}")
print(f"Groupes de doublons exacts              : {len(duplicate_groups)}")
print(f"Occurrences excédentaires à supprimer   : {duplicate_excess_count}")
display(duplicate_groups.head(5))

Lignes appartenant à un groupe répété : 105
Groupes de doublons exacts              : 51
Occurrences excédentaires à supprimer   : 54


,titre,resume,cible,taille_groupe
0,The Beast,Fishing and jobs are becoming scarce in a small seaport community when tragedy strikes a young couple disappear into...,1,3
1,The Bees,Corporate smuggling of South American killer bees into the United States results in huge swarms terrorizing the nort...,1,3
2,Survival Island,"Jack & Jennifer take a private cruise with friends, such as Captain Richards and his wife Maggie. Manuel is a staff ...",1,3
3,Grave Secrets: The Legacy of Hilltop Drive,Jean and Shag Williams locate a newly built house and decide it's perfect for them to buy. One thing the developers ...,1,2
4,Horror Planet,"A group of scientists are working in a lab located on a distant planet. One of them, Sandy, is attacked, raped and i...",1,2


### 1.6 Résumés identiques avec des étiquettes contradictoires

Un même résumé associé à `0` et à `1` rendrait la vérité terrain contradictoire. `nunique()` compte le nombre de cibles différentes pour chaque résumé.

In [6]:
label_count_per_resume = df_raw.groupby("resume")["cible"].nunique()
conflicting_resumes = label_count_per_resume[label_count_per_resume > 1]

print(f"Résumés associés à plusieurs étiquettes : {len(conflicting_resumes)}")
if not conflicting_resumes.empty:
    display(
        df_raw[df_raw["resume"].isin(conflicting_resumes.index)]
        .sort_values(["resume", "cible"])
    )

Résumés associés à plusieurs étiquettes : 0


### 1.7 Nettoyage en mémoire et contrôles

Nous conservons la première occurrence de chaque ligne exacte. Cette opération est faite **avant** le découpage : sinon deux copies d'un même film pourraient se retrouver dans des lots différents et produire une fuite de données. Le fichier CSV original n'est pas réécrit.

In [7]:
df_clean = (
    df_raw.drop_duplicates(subset=EXPECTED_COLUMNS, keep="first")
    .reset_index(drop=True)
    .copy()
)

print(f"Avant nettoyage : {len(df_raw)} lignes")
print(f"Après nettoyage : {len(df_clean)} lignes")
print(f"Lignes retirées  : {len(df_raw) - len(df_clean)}")
display(df_clean["cible"].value_counts().sort_index().rename("effectif").to_frame())

assert df_clean.duplicated(subset=EXPECTED_COLUMNS).sum() == 0
assert df_clean[EXPECTED_COLUMNS].isna().sum().sum() == 0
assert set(df_clean["cible"].unique()) == {0, 1}
assert conflicting_resumes.empty
assert len(df_raw) - len(df_clean) == duplicate_excess_count

print("Contrôles de nettoyage : OK")

Avant nettoyage : 2400 lignes
Après nettoyage : 2346 lignes
Lignes retirées  : 54


,effectif
cible,
0,1195
1,1151


Contrôles de nettoyage : OK


### Résultats attendus avec le fichier fourni

L'exécution doit retrouver 2 400 lignes avant nettoyage, 54 occurrences excédentaires et 2 346 lignes après nettoyage. Aucun résumé ne doit porter deux étiquettes différentes et la dernière cellule doit afficher `Contrôles de nettoyage : OK`.

Questions de compréhension :

1. Pourquoi faut-il retirer les doublons avant de constituer les différents lots ?
2. Quelle différence y a-t-il entre une valeur manquante et une chaîne composée uniquement d'espaces ?

## Partie 2 — Découpage reproductible des données

Objectifs :

- créer un réservoir d'apprentissage de 60 %, un test final de 20 % et un lot de production de 20 % ;
- conserver approximativement la proportion des classes avec la stratification ;
- créer la validation interne uniquement à partir du réservoir d'apprentissage ;
- conserver des identifiants permettant de reconstruire exactement les lots.

Le test final ne servira qu'à l'unique campagne d'évaluation finale. Le lot de production ne servira ni à l'apprentissage, ni au réglage, ni au calcul du score final.

### 2.1 Découpage principal 60/20/20

`random_state` rend le tirage reproductible. `stratify=df["cible"]` demande à scikit-learn de préserver la distribution de la cible. Nous séparons d'abord 60 % et 40 %, puis divisons les 40 % restants en deux moitiés. Comme 2 346 n'est pas divisible exactement selon ces proportions, un écart d'une ligne dû aux arrondis est normal.

In [8]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42

# Identifiant stable dans le jeu nettoyé : il permettra de reconstruire les lots.
df_with_id = df_clean.reset_index(names="clean_row_id")

train_pool, temporary_holdout = train_test_split(
    df_with_id,
    test_size=0.40,
    random_state=SPLIT_SEED,
    stratify=df_with_id["cible"],
)
test_final, production_holdout = train_test_split(
    temporary_holdout,
    test_size=0.50,
    random_state=SPLIT_SEED,
    stratify=temporary_holdout["cible"],
)

# La variable intermédiaire n'est plus utile après la création des deux lots.
del temporary_holdout

### 2.2 Validation interne dans les seuls 60 % d'apprentissage

Nous affectons 20 % du réservoir d'apprentissage à la validation interne. Cela représente environ `0,20 × 0,60 = 0,12`, donc 12 % du jeu complet. Les modèles apprendront d'abord sur environ 48 % des données et seront comparés sur ces 12 %. Une fois les choix figés, le réservoir complet de 60 % pourra être utilisé pour le réentraînement final.

In [9]:
train_internal, validation_internal = train_test_split(
    train_pool,
    test_size=0.20,
    random_state=SPLIT_SEED,
    stratify=train_pool["cible"],
)

### 2.3 Effectifs et absence de chevauchement

Une fuite de données apparaîtrait si le même identifiant se trouvait dans plusieurs lots principaux. Les assertions suivantes vérifient que les lots sont disjoints, qu'aucune ligne n'a été perdue et que la validation interne provient uniquement du réservoir d'apprentissage.

In [10]:
def summarize_splits(split_mapping, total_rows):
    rows = []
    for split_name, frame in split_mapping.items():
        counts = frame["cible"].value_counts().reindex([0, 1], fill_value=0)
        rows.append(
            {
                "lot": split_name,
                "lignes": len(frame),
                "part_du_total": len(frame) / total_rows,
                "comedie_0": int(counts.loc[0]),
                "horreur_1": int(counts.loc[1]),
                "proportion_horreur": frame["cible"].mean(),
            }
        )
    return pd.DataFrame(rows).set_index("lot")

main_splits = {
    "apprentissage_60": train_pool,
    "test_final_20": test_final,
    "production_20": production_holdout,
}
internal_splits = {
    "apprentissage_interne": train_internal,
    "validation_interne": validation_internal,
}

display(summarize_splits(main_splits, len(df_with_id)).round(4))
display(summarize_splits(internal_splits, len(df_with_id)).round(4))

main_id_sets = {
    name: set(frame["clean_row_id"]) for name, frame in main_splits.items()
}
all_clean_ids = set(df_with_id["clean_row_id"])

assert main_id_sets["apprentissage_60"].isdisjoint(main_id_sets["test_final_20"])
assert main_id_sets["apprentissage_60"].isdisjoint(main_id_sets["production_20"])
assert main_id_sets["test_final_20"].isdisjoint(main_id_sets["production_20"])
assert set().union(*main_id_sets.values()) == all_clean_ids

train_internal_ids = set(train_internal["clean_row_id"])
validation_internal_ids = set(validation_internal["clean_row_id"])
assert train_internal_ids.isdisjoint(validation_internal_ids)
assert train_internal_ids | validation_internal_ids == main_id_sets["apprentissage_60"]

print("Contrôles des découpages : OK")

,lignes,part_du_total,comedie_0,horreur_1,proportion_horreur
lot,,,,,
apprentissage_60,1407,0.5997,717,690,0.4904
test_final_20,469,0.1999,239,230,0.4904
production_20,470,0.2003,239,231,0.4915


,lignes,part_du_total,comedie_0,horreur_1,proportion_horreur
lot,,,,,
apprentissage_interne,1125,0.4795,573,552,0.4907
validation_interne,282,0.1202,144,138,0.4894


Contrôles des découpages : OK


### 2.4 Conservation des indices

Nous enregistrons uniquement les identifiants des lignes, la graine et l'empreinte SHA-256 du CSV — pas une nouvelle copie des textes. L'empreinte permet de détecter si le fichier source a changé. Le JSON rendra le découpage vérifiable hors du notebook.

In [11]:
import hashlib
import json

SPLIT_INDEX_PATH = PROJECT_ROOT / "data" / "splits" / "split_indices.json"
SPLIT_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

with DATA_PATH.open("rb") as source_file:
    source_sha256 = hashlib.file_digest(source_file, "sha256").hexdigest()

split_payload = {
    "schema_version": 1,
    "source_file": DATA_PATH.name,
    "source_sha256": source_sha256,
    "split_seed": SPLIT_SEED,
    "internal_validation_fraction_of_train_pool": 0.20,
    "splits": {
        "train_internal": sorted(train_internal_ids),
        "validation_internal": sorted(validation_internal_ids),
        "test_final": sorted(main_id_sets["test_final_20"]),
        "production_holdout": sorted(main_id_sets["production_20"]),
    },
}

with SPLIT_INDEX_PATH.open("w", encoding="utf-8") as output_file:
    json.dump(split_payload, output_file, ensure_ascii=False, indent=2)

print(f"Indices enregistrés : {SPLIT_INDEX_PATH}")
print(f"SHA-256 du CSV       : {source_sha256}")

Indices enregistrés : C:\Users\driou\OneDrive\Bureau\modeleIA\data\splits\split_indices.json
SHA-256 du CSV       : 8f23e70f02dee0abba114cd5988ea20a505b611a1637d027177f5c87fe7cdb19


### Contrôle de fin de partie 2

La cellule de contrôle doit afficher `Contrôles des découpages : OK`. À partir de maintenant, seules `train_internal` et `validation_internal` peuvent servir aux comparaisons et aux réglages.

Questions de compréhension :

1. Pourquoi le second découpage principal utilise-t-il `test_size=0.50` alors que nous voulons 20 % pour chaque lot ?
2. Quelle serait la fuite méthodologique si nous choisissions un hyperparamètre après avoir regardé son score sur `test_final` ?

## Partie 3 — Représentation TF-IDF sans fuite de données

Un algorithme de machine learning ne reçoit pas directement des phrases : il reçoit une matrice numérique. TF-IDF attribue un poids élevé aux termes fréquents dans un document mais relativement rares dans l'ensemble d'apprentissage.

Pour un terme `t` et un document `d` :

- avec `sublinear_tf=True` : `TF(t,d) = 1 + ln(nombre d'occurrences)` lorsque le terme est présent ;
- avec le lissage de scikit-learn : `IDF(t) = ln((1+n)/(1+df(t))) + 1` ;
- avant normalisation : `TF-IDF(t,d) = TF(t,d) × IDF(t)`.

`n` est le nombre de documents d'apprentissage et `df(t)` le nombre de ces documents contenant le terme. Exemple : pour 3 documents et un terme présent dans un seul, `IDF = ln(4/2)+1 ≈ 1,693`. S'il apparaît deux fois dans un document, son TF sous-linéaire vaut aussi `1+ln(2) ≈ 1,693`, donc son poids brut vaut environ `2,867`.

### 3.1 Paramètres du vectoriseur

Les choix ci-dessous constituent un prétraitement de départ commun aux modèles :

- minuscules et suppression des accents Unicode ;
- unigrammes et bigrammes, par exemple `film` et `film drôle` ;
- termes présents dans au moins deux documents et dans au plus 95 % des documents ;
- maximum de 10 000 caractéristiques pour maîtriser la mémoire ;
- normalisation L2 : la racine de la somme des poids au carré vaut 1 pour chaque ligne non vide ;
- nombres `float32`, suffisants ici et deux fois moins volumineux que `float64`.

Nous n'appliquons pas de stemming ou de lemmatisation à ce stade : TF-IDF fournit une référence simple, reproductible et sérialisable sans dépendance linguistique supplémentaire.

In [12]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

TFIDF_PARAMS = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.95,
    "max_features": 10_000,
    "sublinear_tf": True,
    "norm": "l2",
    "dtype": np.float32,
}

tfidf_vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

# fit_transform apprend le vocabulaire et l'IDF sur l'apprentissage autorisé.
X_train_internal = tfidf_vectorizer.fit_transform(train_internal["resume"])

# transform réutilise strictement ce qui vient d'être appris, sans réajustement.
X_validation_internal = tfidf_vectorizer.transform(validation_internal["resume"])

y_train_internal = train_internal["cible"].to_numpy(dtype=np.int32)
y_validation_internal = validation_internal["cible"].to_numpy(dtype=np.int32)

### 3.2 Vocabulaire et dimensions

Dans une matrice TF-IDF, une ligne correspond à un résumé et une colonne à une caractéristique du vocabulaire. Les matrices d'apprentissage et de validation ont donc des nombres de lignes différents, mais exactement les mêmes colonnes. Un terme rencontré uniquement en validation est ignoré puisqu'il ne possède pas de colonne apprise.

In [13]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"Matrice d'apprentissage : {X_train_internal.shape}")
print(f"Matrice de validation    : {X_validation_internal.shape}")
print(f"Taille du vocabulaire   : {len(feature_names)}")
print(f"Type de matrice         : {type(X_train_internal).__name__}")
print(f"Type numérique          : {X_train_internal.dtype}")

assert X_train_internal.shape[0] == len(train_internal)
assert X_validation_internal.shape[0] == len(validation_internal)
assert X_train_internal.shape[1] == X_validation_internal.shape[1]
assert X_train_internal.shape[1] == len(feature_names)
assert tfidf_vectorizer.idf_.shape[0] == len(feature_names)

row_norms = np.sqrt(X_train_internal.multiply(X_train_internal).sum(axis=1)).A1
assert np.allclose(row_norms[row_norms > 0], 1.0, atol=1e-5)
print("Contrôles TF-IDF : OK")

Matrice d'apprentissage : (1125, 10000)
Matrice de validation    : (282, 10000)
Taille du vocabulaire   : 10000
Type de matrice         : csr_matrix
Type numérique          : float32
Contrôles TF-IDF : OK


### 3.3 Pourquoi une matrice creuse ?

Un résumé n'utilise qu'une petite partie du vocabulaire. Une matrice dense mémoriserait aussi tous les zéros. Le format CSR ne stocke essentiellement que les valeurs non nulles, leurs colonnes et la position de début de chaque ligne. Nous calculons ci-dessous le coût réel de la matrice creuse et le coût théorique de sa version dense, sans effectuer la conversion.

In [14]:
def sparse_size_mib(matrix):
    byte_count = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    return byte_count / (1024**2)

def dense_size_mib(matrix):
    byte_count = matrix.shape[0] * matrix.shape[1] * matrix.dtype.itemsize
    return byte_count / (1024**2)

non_zero_count = X_train_internal.nnz
total_cell_count = X_train_internal.shape[0] * X_train_internal.shape[1]
density = non_zero_count / total_cell_count

print(f"Valeurs non nulles       : {non_zero_count:,}")
print(f"Densité                  : {density:.4%}")
print(f"Mémoire CSR mesurée      : {sparse_size_mib(X_train_internal):.2f} Mio")
print(f"Mémoire dense théorique  : {dense_size_mib(X_train_internal):.2f} Mio")

Valeurs non nulles       : 101,216
Densité                  : 0.8997%
Mémoire CSR mesurée      : 0.78 Mio
Mémoire dense théorique  : 42.92 Mio


### 3.4 Exemple des poids d'un résumé

Les poids ci-dessous ne représentent pas encore l'importance pour une classe. Ils indiquent seulement quelles caractéristiques TF-IDF sont les plus présentes dans ce résumé relativement au corpus d'apprentissage.

In [15]:
example_row = X_train_internal[0]
top_positions = np.argsort(example_row.data)[-10:][::-1]
top_feature_indices = example_row.indices[top_positions]

example_weights = pd.DataFrame(
    {
        "caracteristique": feature_names[top_feature_indices],
        "poids_tfidf": example_row.data[top_positions],
    }
)
display(train_internal.iloc[[0]][["titre", "cible", "resume"]])
display(example_weights)

,titre,cible,resume
1728,King of Thorn,1,"A mysterious virus, nicknamed Medusa, is spreading around Japan, turning its victims into stone. Given the impossibi..."


,caracteristique,poids_tfidf
0,bed,0.202764
1,of finding,0.142326
2,monstrous creatures,0.142326
3,nicknamed,0.142326
4,medusa,0.142326
5,manga,0.142326
6,in wait,0.136415
7,years and,0.136415
8,explosive,0.136415
9,sensational,0.136415


### Contrôle de fin de partie 3

La cellule de dimensions doit afficher `Contrôles TF-IDF : OK`. Nous n'avons créé volontairement aucune matrice pour `test_final` ou `production_holdout` : ces lots restent réservés.

Questions de compréhension :

1. Pourquoi utilisons-nous `fit_transform` sur l'apprentissage mais seulement `transform` sur la validation ?
2. Pourquoi la conversion immédiate de toute la matrice TF-IDF en tableau dense est-elle généralement une mauvaise idée ?

## Partie 4 — Modèles de référence sur cinq graines

Nous comparons trois familles dans des conditions communes : mêmes textes nettoyés, même TF-IDF, mêmes lignes d'apprentissage et même validation interne. Aucun résultat du test final ou du lot de production n'est consulté.

Pour la classe positive `1 = horreur` :

- `accuracy = (VP + VN) / N` : proportion totale de prédictions correctes ;
- `précision = VP / (VP + FP)` : parmi les films prédits horreur, proportion réellement horreur ;
- `rappel = VP / (VP + FN)` : parmi les films réellement horreur, proportion retrouvée ;
- `F1 = 2 × précision × rappel / (précision + rappel)` : moyenne harmonique de la précision et du rappel.

`VP`, `VN`, `FP` et `FN` désignent les vrais positifs, vrais négatifs, faux positifs et faux négatifs. Le F1 est le critère principal demandé.

### 4.1 Intuition des trois modèles

**Régression logistique.** Elle calcule un score linéaire `z = b + w·x`, puis une probabilité avec la sigmoïde `p = 1/(1+exp(-z))`. Chaque poids mesure l'influence d'une caractéristique TF-IDF sur la classe horreur. Malgré son nom, c'est ici un classifieur.

**Bayes naïf multinomial.** Il estime la fréquence des caractéristiques dans chaque classe puis compare les probabilités des classes. L'hypothèse « naïve » suppose les caractéristiques conditionnellement indépendantes connaissant la classe. Cette hypothèse est imparfaite pour le langage, mais le modèle est souvent très compétitif sur des comptes ou poids de termes. `alpha=1` applique un lissage pour éviter les probabilités nulles.

**Réseau dense.** Chaque neurone calcule une combinaison pondérée suivie d'une activation. ReLU utilise `max(0,z)` dans les couches cachées ; la sigmoïde finale renvoie une probabilité entre 0 et 1. L'entropie croisée binaire pénalise les probabilités éloignées de la vraie classe. La rétropropagation calcule les gradients de cette perte et Adam met à jour les poids.

Le dropout désactive aléatoirement une partie des activations pendant l'apprentissage. La régularisation L2 pénalise les poids trop grands. L'early stopping arrête l'entraînement lorsque la perte de validation ne s'améliore plus et restaure les meilleurs poids. Ces mécanismes limitent le surapprentissage.

### 4.2 Évaluation commune

Nous gardons le seuil de décision standard `0,5` et cinq graines fixes. Les graines contrôlent les opérations aléatoires ; elles ne créent pas cinq nouveaux jeux de données. Bayes naïf est déterministe et la régression logistique utilisée ici devrait être stable : leur répétition sert surtout à rendre explicite cette absence de variabilité.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.naive_bayes import MultinomialNB

MODEL_SEEDS = [11, 22, 33, 44, 55]
DECISION_THRESHOLD = 0.50

def binary_classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

run_records = []

In [17]:
for seed in MODEL_SEEDS:
    logistic_model = LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=2_000,
        random_state=seed,
    )
    logistic_model.fit(X_train_internal, y_train_internal)
    logistic_predictions = logistic_model.predict(X_validation_internal)
    run_records.append(
        {
            "model": "regression_logistique",
            "seed": seed,
            "epochs": np.nan,
            **binary_classification_metrics(y_validation_internal, logistic_predictions),
        }
    )

    naive_bayes_model = MultinomialNB(alpha=1.0)
    naive_bayes_model.fit(X_train_internal, y_train_internal)
    naive_bayes_predictions = naive_bayes_model.predict(X_validation_internal)
    run_records.append(
        {
            "model": "bayes_naif_multinomial",
            "seed": seed,
            "epochs": np.nan,
            **binary_classification_metrics(y_validation_internal, naive_bayes_predictions),
        }
    )

print("Modèles classiques entraînés sur cinq graines : OK")

Modèles classiques entraînés sur cinq graines : OK


### 4.3 Petit réseau dense de référence

Keras attend ici un tableau dense. Nous effectuons maintenant la conversion contrôlée dont le coût a été mesuré : environ 42,92 Mio pour l'apprentissage et 10,76 Mio pour la validation. La fonction de construction accepte déjà une liste de couches, le dropout, la régularisation et le taux d'apprentissage ; elle pourra donc être réutilisée sans changer l'architecture conceptuelle lors des optimisations.

In [18]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf

X_train_dense = X_train_internal.toarray()
X_validation_dense = X_validation_internal.toarray()

def build_dense_network(
    input_dim,
    hidden_units,
    dropout_rate,
    l2_strength,
    learning_rate,
):
    model = tf.keras.Sequential(name="tfidf_dense_classifier")
    model.add(tf.keras.layers.Input(shape=(input_dim,), name="tfidf_input"))

    for layer_number, unit_count in enumerate(hidden_units, start=1):
        model.add(
            tf.keras.layers.Dense(
                unit_count,
                activation="relu",
                kernel_regularizer=tf.keras.regularizers.l2(l2_strength),
                name=f"dense_{layer_number}",
            )
        )
        model.add(tf.keras.layers.Dropout(dropout_rate, name=f"dropout_{layer_number}"))

    model.add(tf.keras.layers.Dense(1, activation="sigmoid", name="probability"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.F1Score(
                name="f1", average="micro", threshold=DECISION_THRESHOLD
            )
        ],
    )
    return model

print(f"Apprentissage dense : {X_train_dense.shape}, {X_train_dense.nbytes / 1024**2:.2f} Mio")
print(f"Validation dense    : {X_validation_dense.shape}, {X_validation_dense.nbytes / 1024**2:.2f} Mio")

Apprentissage dense : (1125, 10000), 42.92 Mio
Validation dense    : (282, 10000), 10.76 Mio


In [19]:
BASELINE_NETWORK_PARAMS = {
    "hidden_units": (64,),
    "dropout_rate": 0.30,
    "l2_strength": 1e-4,
    "learning_rate": 1e-3,
}

for seed in MODEL_SEEDS:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    neural_model = build_dense_network(
        input_dim=X_train_dense.shape[1],
        **BASELINE_NETWORK_PARAMS,
    )
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=2,
        min_delta=1e-4,
        restore_best_weights=True,
    )
    history = neural_model.fit(
        X_train_dense,
        y_train_internal.reshape(-1, 1),
        validation_data=(X_validation_dense, y_validation_internal.reshape(-1, 1)),
        epochs=20,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=0,
    )
    neural_probabilities = neural_model.predict(X_validation_dense, verbose=0).ravel()
    neural_predictions = (neural_probabilities >= DECISION_THRESHOLD).astype(np.int32)
    run_records.append(
        {
            "model": "reseau_dense_reference",
            "seed": seed,
            "epochs": len(history.history["loss"]),
            **binary_classification_metrics(y_validation_internal, neural_predictions),
        }
    )

print(f"Paramètres entraînables du réseau : {neural_model.count_params():,}")
print("Réseau entraîné sur cinq graines : OK")


Paramètres entraînables du réseau : 640,129
Réseau entraîné sur cinq graines : OK


### 4.4 Résultats individuels, moyenne et dispersion

La moyenne résume la performance observée sur les cinq exécutions. L'écart-type mesure leur dispersion : plus il est élevé, plus le résultat dépend de la graine. Nous ne retenons pas simplement la meilleure graine, car ce serait une sélection opportuniste sur la validation.

In [20]:
validation_runs = pd.DataFrame(run_records)
validation_summary = (
    validation_runs.groupby("model")
    .agg(
        runs=("seed", "count"),
        accuracy_mean=("accuracy", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .sort_values("f1_mean", ascending=False)
)

display(validation_runs.sort_values(["model", "seed"]).round(4))
display(validation_summary.round(4))

assert validation_runs.groupby("model").size().eq(5).all()
assert validation_runs[["accuracy", "precision", "recall", "f1"]].notna().all().all()
assert validation_runs[["accuracy", "precision", "recall", "f1"]].apply(
    lambda column: column.between(0, 1).all()
).all()

RESULTS_DIR = PROJECT_ROOT / "evidence" / "metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
validation_runs.to_csv(
    RESULTS_DIR / "baseline_validation_runs.csv", index=False, encoding="utf-8"
)
validation_summary.to_csv(
    RESULTS_DIR / "baseline_validation_summary.csv", encoding="utf-8"
)
print("Contrôles des modèles de référence : OK")

,model,seed,epochs,accuracy,precision,recall,f1
1,bayes_naif_multinomial,11,NaN,0.8475,0.8800,0.7971,0.8365
3,bayes_naif_multinomial,22,NaN,0.8475,0.8800,0.7971,0.8365
5,bayes_naif_multinomial,33,NaN,0.8475,0.8800,0.7971,0.8365
7,bayes_naif_multinomial,44,NaN,0.8475,0.8800,0.7971,0.8365
9,bayes_naif_multinomial,55,NaN,0.8475,0.8800,0.7971,0.8365
0,regression_logistique,11,NaN,0.8369,0.8594,0.7971,0.8271
2,regression_logistique,22,NaN,0.8369,0.8594,0.7971,0.8271
4,regression_logistique,33,NaN,0.8369,0.8594,0.7971,0.8271
6,regression_logistique,44,NaN,0.8369,0.8594,0.7971,0.8271
8,regression_logistique,55,NaN,0.8369,0.8594,0.7971,0.8271


,runs,accuracy_mean,precision_mean,recall_mean,f1_mean,f1_std
model,,,,,,
reseau_dense_reference,5,0.8780,0.8961,0.8493,0.8720,0.0069
bayes_naif_multinomial,5,0.8475,0.8800,0.7971,0.8365,0.0000
regression_logistique,5,0.8369,0.8594,0.7971,0.8271,0.0000


Contrôles des modèles de référence : OK


### 4.5 Marge de bruit pédagogique demandée

Le brief impose le calcul `1,96 × sqrt(p × (1-p) / n)`, avec `p` le score observé et `n` la taille de validation. Cette formule ressemble à la marge d'un intervalle binomial pour une proportion. Or le F1 est un rapport non linéaire construit à partir de la précision et du rappel : cette valeur n'est donc **pas automatiquement un intervalle de confiance statistiquement valide pour le F1**. Nous l'affichons comme ordre de grandeur pédagogique et l'interprétons avec l'écart-type entre graines.

In [21]:
validation_size = len(y_validation_internal)
validation_summary["marge_pedagogique_f1"] = 1.96 * np.sqrt(
    validation_summary["f1_mean"]
    * (1 - validation_summary["f1_mean"])
    / validation_size
)
display(validation_summary[["f1_mean", "f1_std", "marge_pedagogique_f1"]].round(4))

,f1_mean,f1_std,marge_pedagogique_f1
model,,,
reseau_dense_reference,0.8720,0.0069,0.0390
bayes_naif_multinomial,0.8365,0.0000,0.0432
regression_logistique,0.8271,0.0000,0.0441


### Contrôle de fin de partie 4

La dernière cellule de résultats doit afficher `Contrôles des modèles de référence : OK`. Le meilleur F1 moyen sur la validation interne est un résultat provisoire : le test final reste fermé et l'optimisation du réseau n'a pas encore été réalisée.

Questions de compréhension :

1. Si Bayes naïf obtient exactement le même F1 pour les cinq graines, est-ce une erreur ? Pourquoi ?
2. Pourquoi comparons-nous la moyenne et l'écart-type plutôt que de retenir uniquement la meilleure exécution du réseau ?

## Partie 5 — Optimisation avec KerasTuner

KerasTuner automatise la proposition d'hyperparamètres, l'entraînement et la comparaison des essais. Nous utilisons `RandomSearch` : 20 configurations sont tirées pseudo-aléatoirement dans un domaine fixé **avant** de regarder les résultats.

Un paramètre est appris par rétropropagation, par exemple un poids du réseau. Un hyperparamètre organise cet apprentissage, par exemple le nombre de couches ou le taux d'apprentissage.

Les 20 essais partagent les mêmes données, le même TF-IDF, le même seuil de décision, un maximum de 15 époques, un batch de 64 et le F1 de validation comme objectif. Un seul entraînement est réalisé par essai ; la robustesse sur cinq graines sera réévaluée après le choix des configurations.

### 5.1 Espace de recherche

L'espace contient `3 × 3 × 3 × 3 × 4 = 324` combinaisons possibles. Avec seulement 20 essais, la recherche n'est pas exhaustive : son résultat est le meilleur point observé dans ce budget, pas une preuve d'optimalité globale. Pour faciliter une comparaison équitable avec Optuna, toutes les couches cachées d'un essai utilisent le même nombre de neurones.

In [22]:
import keras_tuner as kt

TUNING_TRIALS = 20
TUNING_EPOCHS = 15
TUNING_BATCH_SIZE = 64
TUNING_SEED = 2026

HYPERPARAMETER_SPACE = {
    "num_layers": [1, 2, 3],
    "units_per_layer": [32, 64, 128],
    "dropout_rate": [0.10, 0.30, 0.50],
    "l2_strength": [1e-5, 1e-4, 1e-3],
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3],
}

def build_keras_tuner_model(hp):
    num_layers = hp.Choice("num_layers", HYPERPARAMETER_SPACE["num_layers"])
    units_per_layer = hp.Choice(
        "units_per_layer", HYPERPARAMETER_SPACE["units_per_layer"]
    )
    dropout_rate = hp.Choice(
        "dropout_rate", HYPERPARAMETER_SPACE["dropout_rate"]
    )
    l2_strength = hp.Choice(
        "l2_strength", HYPERPARAMETER_SPACE["l2_strength"]
    )
    learning_rate = hp.Choice(
        "learning_rate", HYPERPARAMETER_SPACE["learning_rate"]
    )

    return build_dense_network(
        input_dim=X_train_dense.shape[1],
        hidden_units=(units_per_layer,) * num_layers,
        dropout_rate=dropout_rate,
        l2_strength=l2_strength,
        learning_rate=learning_rate,
    )

### 5.2 Recherche aléatoire de 20 essais

`overwrite=True` recrée cette expérience locale avec les mêmes règles lorsque le notebook est relancé. Le dossier de travail de KerasTuner est exclu de Git ; les tableaux de résultats synthétiques seront, eux, conservés dans `evidence/metrics`.

In [23]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(TUNING_SEED)

keras_tuner_search = kt.RandomSearch(
    hypermodel=build_keras_tuner_model,
    objective=kt.Objective("val_f1", direction="max"),
    max_trials=TUNING_TRIALS,
    executions_per_trial=1,
    seed=TUNING_SEED,
    directory=str(PROJECT_ROOT / "keras_tuner"),
    project_name="dense_f1_random_search",
    overwrite=True,
)

tuning_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_f1",
    mode="max",
    patience=2,
    min_delta=1e-3,
    restore_best_weights=True,
)

keras_tuner_search.search(
    X_train_dense,
    y_train_internal.reshape(-1, 1),
    validation_data=(X_validation_dense, y_validation_internal.reshape(-1, 1)),
    epochs=TUNING_EPOCHS,
    batch_size=TUNING_BATCH_SIZE,
    callbacks=[tuning_early_stopping],
    verbose=0,
)

completed_keras_trials = [
    trial
    for trial in keras_tuner_search.oracle.trials.values()
    if trial.status == "COMPLETED"
]
print(f"Essais KerasTuner terminés : {len(completed_keras_trials)}/{TUNING_TRIALS}")
assert len(completed_keras_trials) == TUNING_TRIALS

Essais KerasTuner terminés : 20/20


### 5.3 Résultats et vérification indépendante du meilleur essai

Nous récupérons tous les essais terminés, puis rechargeons le meilleur réseau. Son F1 est recalculé avec la fonction scikit-learn déjà utilisée pour les références : cette seconde mesure vérifie que l'objectif de KerasTuner et notre tableau de comparaison ont la même signification.

In [24]:
keras_trial_rows = []
for trial in completed_keras_trials:
    keras_trial_rows.append(
        {
            "trial_id": trial.trial_id,
            "val_f1": float(trial.score),
            **trial.hyperparameters.values,
        }
    )

keras_tuner_trials = (
    pd.DataFrame(keras_trial_rows)
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)
display(keras_tuner_trials.round(5))

best_keras_hp = keras_tuner_search.get_best_hyperparameters(num_trials=1)[0]
best_keras_trial = keras_tuner_search.oracle.get_best_trials(num_trials=1)[0]
best_keras_model = keras_tuner_search.get_best_models(num_models=1)[0]

best_keras_probabilities = best_keras_model.predict(X_validation_dense, verbose=0).ravel()
best_keras_predictions = (
    best_keras_probabilities >= DECISION_THRESHOLD
).astype(np.int32)
best_keras_metrics = binary_classification_metrics(
    y_validation_internal, best_keras_predictions
)

print("Meilleurs hyperparamètres KerasTuner :")
print(best_keras_hp.values)
print(f"F1 enregistré par KerasTuner : {best_keras_trial.score:.4f}")
print(f"F1 recalculé avec scikit-learn : {best_keras_metrics['f1']:.4f}")

,trial_id,val_f1,num_layers,units_per_layer,dropout_rate,l2_strength,learning_rate
0,07,0.88055,2,128,0.5,0.00100,0.0010
1,04,0.87500,2,64,0.1,0.00010,0.0010
2,17,0.87365,2,32,0.3,0.00001,0.0030
3,01,0.87197,2,128,0.3,0.00001,0.0030
4,10,0.87179,1,64,0.5,0.00010,0.0001
5,16,0.86957,2,32,0.1,0.00100,0.0003
6,06,0.86869,3,128,0.1,0.00010,0.0010
7,14,0.86765,1,32,0.1,0.00100,0.0030
8,13,0.86245,1,128,0.3,0.00001,0.0010
9,09,0.85606,1,64,0.5,0.00010,0.0030


c:\Users\driou\OneDrive\Bureau\modeleIA\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


Meilleurs hyperparamètres KerasTuner :
{'num_layers': 2, 'units_per_layer': 128, 'dropout_rate': 0.5, 'l2_strength': 0.001, 'learning_rate': 0.001}
F1 enregistré par KerasTuner : 0.8805
F1 recalculé avec scikit-learn : 0.8805


In [25]:
keras_tuner_trials.to_csv(
    RESULTS_DIR / "keras_tuner_trials.csv", index=False, encoding="utf-8"
)

keras_tuner_evidence = {
    "tool": "KerasTuner RandomSearch",
    "objective": "val_f1",
    "trial_budget": TUNING_TRIALS,
    "epochs_per_trial_max": TUNING_EPOCHS,
    "batch_size": TUNING_BATCH_SIZE,
    "seed": TUNING_SEED,
    "train_rows": len(y_train_internal),
    "validation_rows": len(y_validation_internal),
    "search_space": HYPERPARAMETER_SPACE,
    "best_hyperparameters": best_keras_hp.values,
    "best_trial_val_f1": float(best_keras_trial.score),
    "recomputed_validation_metrics": {
        name: float(value) for name, value in best_keras_metrics.items()
    },
}
with (RESULTS_DIR / "keras_tuner_best.json").open("w", encoding="utf-8") as file:
    json.dump(keras_tuner_evidence, file, ensure_ascii=False, indent=2)

assert np.isclose(best_keras_trial.score, best_keras_metrics["f1"], atol=1e-5)
assert set(best_keras_hp.values) == set(HYPERPARAMETER_SPACE)
print("Contrôles KerasTuner : OK")

Contrôles KerasTuner : OK


### Contrôle de fin de partie 5

Le résultat attendu est `Essais KerasTuner terminés : 20/20`, puis `Contrôles KerasTuner : OK`. Le meilleur essai reste une observation sur la validation interne ; nous ne le déclarons pas encore modèle final.

Questions de compréhension :

1. Pourquoi 20 essais aléatoires ne garantissent-ils pas d'avoir trouvé la meilleure des 324 configurations ?
2. Pourquoi ne relançons-nous pas maintenant le meilleur essai jusqu'à obtenir une graine encore plus favorable ?

## Partie 6 — Optimisation avec Optuna

KerasTuner est spécialisé dans les modèles Keras et organise directement la construction, l'entraînement et les checkpoints. Optuna est plus général : une fonction `objective(trial)` décrit librement l'expérience et renvoie la valeur à maximiser ou minimiser.

Pour rendre la comparaison défendable, Optuna réutilise :

- la même fonction `build_dense_network` ;
- les cinq mêmes domaines d'hyperparamètres ;
- les mêmes 1 125 lignes d'apprentissage et 282 lignes de validation ;
- le même F1 de la classe horreur ;
- 20 essais, 15 époques maximum, des batchs de 64 et le même early stopping.

La différence étudiée est la stratégie : `RandomSearch` pour KerasTuner et `TPESampler` pour Optuna. Après cinq essais initiaux, TPE compare statistiquement les zones ayant donné de bons et de moins bons résultats afin de proposer des configurations prometteuses. Il ne garantit toutefois pas l'optimum.

### 6.1 Fonction objectif commune

Chaque essai reçoit une graine reproductible dérivée de son numéro. L'objective construit le réseau, l'entraîne, restaure les meilleurs poids observés par l'early stopping, puis recalcule le F1 avec scikit-learn. Nous conservons en mémoire les poids du meilleur essai pour vérifier ensuite exactement son résultat. Aucun pruning Optuna supplémentaire n'est activé afin de garder le même mécanisme d'arrêt que KerasTuner.

In [26]:
required_names_for_optuna = [
    "X_train_dense",
    "X_validation_dense",
    "y_train_internal",
    "y_validation_internal",
    "build_dense_network",
    "binary_classification_metrics",
    "HYPERPARAMETER_SPACE",
    "TUNING_TRIALS",
    "TUNING_EPOCHS",
    "TUNING_BATCH_SIZE",
    "best_keras_metrics",
    "RESULTS_DIR",
]
missing_names = [
    name for name in required_names_for_optuna if name not in globals()
]
assert not missing_names, (
    "Variables manquantes : "
    + ", ".join(missing_names)
    + ". Exécutez les parties 1 à 5 avant la partie 6."
)
print("Pré-requis Optuna : OK")

Pré-requis Optuna : OK


In [27]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
OPTUNA_SEED = TUNING_SEED
optuna_best_state = {"value": -np.inf}

def optuna_objective(trial):
    num_layers = trial.suggest_categorical(
        "num_layers", HYPERPARAMETER_SPACE["num_layers"]
    )
    units_per_layer = trial.suggest_categorical(
        "units_per_layer", HYPERPARAMETER_SPACE["units_per_layer"]
    )
    dropout_rate = trial.suggest_categorical(
        "dropout_rate", HYPERPARAMETER_SPACE["dropout_rate"]
    )
    l2_strength = trial.suggest_categorical(
        "l2_strength", HYPERPARAMETER_SPACE["l2_strength"]
    )
    learning_rate = trial.suggest_categorical(
        "learning_rate", HYPERPARAMETER_SPACE["learning_rate"]
    )

    model_seed = OPTUNA_SEED + trial.number
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(model_seed)

    model = build_dense_network(
        input_dim=X_train_dense.shape[1],
        hidden_units=(units_per_layer,) * num_layers,
        dropout_rate=dropout_rate,
        l2_strength=l2_strength,
        learning_rate=learning_rate,
    )
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_f1",
        mode="max",
        patience=2,
        min_delta=1e-3,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train_dense,
        y_train_internal.reshape(-1, 1),
        validation_data=(X_validation_dense, y_validation_internal.reshape(-1, 1)),
        epochs=TUNING_EPOCHS,
        batch_size=TUNING_BATCH_SIZE,
        callbacks=[early_stopping],
        verbose=0,
    )

    probabilities = model.predict(X_validation_dense, verbose=0).ravel()
    predictions = (probabilities >= DECISION_THRESHOLD).astype(np.int32)
    metrics = binary_classification_metrics(y_validation_internal, predictions)

    trial.set_user_attr("epochs", len(history.history["loss"]))
    trial.set_user_attr("model_seed", model_seed)
    for metric_name, metric_value in metrics.items():
        trial.set_user_attr(metric_name, float(metric_value))

    if metrics["f1"] > optuna_best_state["value"]:
        optuna_best_state.update(
            {
                "value": float(metrics["f1"]),
                "trial_number": trial.number,
                "weights": [weight.copy() for weight in model.get_weights()],
            }
        )

    return float(metrics["f1"])

c:\Users\driou\OneDrive\Bureau\modeleIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 6.2 Étude TPE de 20 essais

L'étude reste en mémoire pendant l'exécution du notebook. Les résultats nécessaires à la preuve seront exportés ensuite en CSV et JSON. `n_jobs=1` évite que plusieurs réseaux se disputent la mémoire du poste.

In [28]:
optuna_sampler = optuna.samplers.TPESampler(
    seed=OPTUNA_SEED,
    n_startup_trials=5,
)
optuna_study = optuna.create_study(
    study_name="dense_f1_tpe_search",
    direction="maximize",
    sampler=optuna_sampler,
    pruner=optuna.pruners.NopPruner(),
)
optuna_study.optimize(
    optuna_objective,
    n_trials=TUNING_TRIALS,
    n_jobs=1,
    gc_after_trial=True,
    show_progress_bar=False,
)

completed_optuna_trials = [
    trial
    for trial in optuna_study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]
print(f"Essais Optuna terminés : {len(completed_optuna_trials)}/{TUNING_TRIALS}")
assert len(completed_optuna_trials) == TUNING_TRIALS

Essais Optuna terminés : 20/20


### 6.3 Résultats et reconstruction du meilleur essai

La reconstruction utilise les hyperparamètres du meilleur essai et les poids copiés à la fin de cet essai. Elle ne constitue donc pas un nouvel entraînement favorable : elle vérifie la prédiction du modèle effectivement évalué.

In [29]:
optuna_trial_rows = []
for trial in completed_optuna_trials:
    optuna_trial_rows.append(
        {
            "trial_id": trial.number,
            "val_f1": float(trial.value),
            "epochs": trial.user_attrs["epochs"],
            "model_seed": trial.user_attrs["model_seed"],
            "accuracy": trial.user_attrs["accuracy"],
            "precision": trial.user_attrs["precision"],
            "recall": trial.user_attrs["recall"],
            **trial.params,
        }
    )

optuna_trials = (
    pd.DataFrame(optuna_trial_rows)
    .sort_values("val_f1", ascending=False)
    .reset_index(drop=True)
)
display(optuna_trials.round(5))

best_optuna_params = optuna_study.best_params
tf.keras.backend.clear_session()
best_optuna_model = build_dense_network(
    input_dim=X_train_dense.shape[1],
    hidden_units=(best_optuna_params["units_per_layer"],)
    * best_optuna_params["num_layers"],
    dropout_rate=best_optuna_params["dropout_rate"],
    l2_strength=best_optuna_params["l2_strength"],
    learning_rate=best_optuna_params["learning_rate"],
)
best_optuna_model.set_weights(optuna_best_state["weights"])
best_optuna_probabilities = best_optuna_model.predict(
    X_validation_dense, verbose=0
).ravel()
best_optuna_predictions = (
    best_optuna_probabilities >= DECISION_THRESHOLD
).astype(np.int32)
best_optuna_metrics = binary_classification_metrics(
    y_validation_internal, best_optuna_predictions
)

print("Meilleurs hyperparamètres Optuna :")
print(best_optuna_params)
print(f"F1 enregistré par Optuna : {optuna_study.best_value:.4f}")
print(f"F1 recalculé avec scikit-learn : {best_optuna_metrics['f1']:.4f}")

,trial_id,val_f1,epochs,model_seed,accuracy,precision,recall,num_layers,units_per_layer,dropout_rate,l2_strength,learning_rate
0,13,0.88889,5,2039,0.89362,0.90909,0.86957,3,64,0.5,0.00100,0.0030
1,5,0.88727,7,2031,0.89007,0.89051,0.88406,2,64,0.5,0.00001,0.0030
2,11,0.88000,9,2037,0.88298,0.88321,0.87681,1,64,0.5,0.00001,0.0030
3,12,0.87912,5,2038,0.88298,0.88889,0.86957,2,64,0.5,0.00010,0.0030
4,2,0.87823,6,2028,0.88298,0.89474,0.86232,1,64,0.5,0.00001,0.0030
5,15,0.87770,5,2041,0.87943,0.87143,0.88406,3,64,0.5,0.00100,0.0030
6,10,0.87681,7,2036,0.87943,0.87681,0.87681,2,32,0.5,0.00001,0.0030
7,19,0.87591,8,2045,0.87943,0.88235,0.86957,3,32,0.5,0.00010,0.0030
8,16,0.86989,11,2042,0.87589,0.89313,0.84783,2,64,0.5,0.00001,0.0010
9,4,0.86765,6,2030,0.87234,0.88060,0.85507,1,32,0.1,0.00100,0.0010


Meilleurs hyperparamètres Optuna :
{'num_layers': 3, 'units_per_layer': 64, 'dropout_rate': 0.5, 'l2_strength': 0.001, 'learning_rate': 0.003}
F1 enregistré par Optuna : 0.8889
F1 recalculé avec scikit-learn : 0.8889


### 6.4 Comparaison équitable et preuves

Nous pouvons comparer le meilleur résultat observé dans chaque budget, mais pas conclure qu'un outil est universellement supérieur à partir d'une seule étude de 20 essais. La stratégie, les configurations effectivement visitées et l'aléa d'initialisation influencent les deux maximums.

In [30]:
optimizer_comparison = pd.DataFrame(
    [
        {
            "outil": "KerasTuner RandomSearch",
            "essais": TUNING_TRIALS,
            "meilleur_f1_validation": float(best_keras_metrics["f1"]),
        },
        {
            "outil": "Optuna TPESampler",
            "essais": TUNING_TRIALS,
            "meilleur_f1_validation": float(best_optuna_metrics["f1"]),
        },
    ]
)
display(optimizer_comparison.round(5))

optuna_trials.to_csv(
    RESULTS_DIR / "optuna_trials.csv", index=False, encoding="utf-8"
)
optimizer_comparison.to_csv(
    RESULTS_DIR / "optimizer_validation_comparison.csv",
    index=False,
    encoding="utf-8",
)

optuna_evidence = {
    "tool": "Optuna TPESampler",
    "objective": "val_f1",
    "trial_budget": TUNING_TRIALS,
    "startup_trials": 5,
    "epochs_per_trial_max": TUNING_EPOCHS,
    "batch_size": TUNING_BATCH_SIZE,
    "seed": OPTUNA_SEED,
    "train_rows": len(y_train_internal),
    "validation_rows": len(y_validation_internal),
    "search_space": HYPERPARAMETER_SPACE,
    "best_trial_number": optuna_study.best_trial.number,
    "best_hyperparameters": best_optuna_params,
    "best_trial_val_f1": float(optuna_study.best_value),
    "recomputed_validation_metrics": {
        name: float(value) for name, value in best_optuna_metrics.items()
    },
}
with (RESULTS_DIR / "optuna_best.json").open("w", encoding="utf-8") as file:
    json.dump(optuna_evidence, file, ensure_ascii=False, indent=2)

assert optuna_best_state["trial_number"] == optuna_study.best_trial.number
assert np.isclose(optuna_study.best_value, best_optuna_metrics["f1"], atol=1e-12)
assert set(best_optuna_params) == set(HYPERPARAMETER_SPACE)
print("Contrôles Optuna : OK")

,outil,essais,meilleur_f1_validation
0,KerasTuner RandomSearch,20,0.88055
1,Optuna TPESampler,20,0.88889


Contrôles Optuna : OK


### Contrôle de fin de partie 6

Le résultat attendu est `Essais Optuna terminés : 20/20`, puis `Contrôles Optuna : OK`. La partie suivante figera les choix sur la validation, évaluera leur stabilité sur cinq graines et définira la règle d'export avant d'ouvrir le test final.

Questions de compréhension :

1. Pourquoi le meilleur F1 d'Optuna et celui de KerasTuner ne suffisent-ils pas, à eux seuls, pour déclarer un outil supérieur ?
2. Quel avantage offre une fonction `objective` explicite dans Optuna par rapport à une recherche étroitement liée à Keras ?

## Partie 7A — Stabilité et décision avant le test final

Cette sous-partie reste entièrement dans l'apprentissage et la validation internes. Elle ne transforme pas et n'évalue pas `test_final`.

La règle de sélection est enregistrée séparément dans `config/model_selection_rule.json` et doit être commitée avant cette expérience. Nous réentraînons les architectures gagnantes de KerasTuner et d'Optuna avec les cinq graines déjà fixées. La décision reposera sur le F1 moyen, sa dispersion et une règle de départage écrite à l'avance.

In [37]:
required_names_for_selection = [
    "X_train_dense",
    "X_validation_dense",
    "y_train_internal",
    "y_validation_internal",
    "build_dense_network",
    "binary_classification_metrics",
    "best_keras_hp",
    "best_optuna_params",
    "MODEL_SEEDS",
    "TUNING_EPOCHS",
    "TUNING_BATCH_SIZE",
    "RESULTS_DIR",
]
missing_names = [
    name for name in required_names_for_selection if name not in globals()
]
assert not missing_names, (
    "Variables manquantes : "
    + ", ".join(missing_names)
    + ". Exécutez les parties précédentes avant la partie 7A."
)
print("Pré-requis de sélection : OK")

Pré-requis de sélection : OK


### 7.1 Lecture de la règle figée

Si les F1 moyens diffèrent de plus de 0,005, la meilleure moyenne gagne. Dans le cas contraire, les performances sont considérées pratiquement proches et nous privilégions le plus petit écart-type. Une égalité restante est départagée par le nombre de paramètres, puis par le nom.

In [38]:
MODEL_SELECTION_RULE_PATH = (
    PROJECT_ROOT / "config" / "model_selection_rule.json"
)
with MODEL_SELECTION_RULE_PATH.open("r", encoding="utf-8") as file:
    model_selection_rule = json.load(file)

ROBUSTNESS_SEEDS = model_selection_rule["model_seeds"]
PRACTICAL_TIE_TOLERANCE = model_selection_rule["practical_tie_tolerance"]

assert ROBUSTNESS_SEEDS == MODEL_SEEDS
assert model_selection_rule["selection_data"] == "validation_internal_only"
assert model_selection_rule["status"] == "frozen_before_final_test"

print(f"Règle chargée : {MODEL_SELECTION_RULE_PATH}")
print(f"Graines       : {ROBUSTNESS_SEEDS}")
print(f"Tolérance     : {PRACTICAL_TIE_TOLERANCE:.3f}")

Règle chargée : C:\Users\driou\OneDrive\Bureau\modeleIA\config\model_selection_rule.json
Graines       : [11, 22, 33, 44, 55]
Tolérance     : 0.005


### 7.2 Deux architectures candidates sur cinq graines

Chaque configuration conserve ses hyperparamètres optimisés, les mêmes données, 15 époques maximum et le même early stopping. Le nombre de meilleures époques est relevé pour fixer ensuite une durée de réentraînement sans validation.

In [39]:
optimized_candidates = {
    "keras_tuner_optimized_network": dict(best_keras_hp.values),
    "optuna_optimized_network": dict(best_optuna_params),
}
assert set(optimized_candidates) == set(model_selection_rule["candidate_scope"])
display(pd.DataFrame(optimized_candidates).T)

,num_layers,units_per_layer,dropout_rate,l2_strength,learning_rate
keras_tuner_optimized_network,2.0,128.0,0.5,0.001,0.001
optuna_optimized_network,3.0,64.0,0.5,0.001,0.003


In [40]:
optimized_run_records = []

for candidate_name, candidate_params in optimized_candidates.items():
    for seed in ROBUSTNESS_SEEDS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)

        candidate_model = build_dense_network(
            input_dim=X_train_dense.shape[1],
            hidden_units=(candidate_params["units_per_layer"],)
            * candidate_params["num_layers"],
            dropout_rate=candidate_params["dropout_rate"],
            l2_strength=candidate_params["l2_strength"],
            learning_rate=candidate_params["learning_rate"],
        )
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_f1",
            mode="max",
            patience=2,
            min_delta=1e-3,
            restore_best_weights=True,
        )
        history = candidate_model.fit(
            X_train_dense,
            y_train_internal.reshape(-1, 1),
            validation_data=(
                X_validation_dense,
                y_validation_internal.reshape(-1, 1),
            ),
            epochs=TUNING_EPOCHS,
            batch_size=TUNING_BATCH_SIZE,
            callbacks=[early_stopping],
            verbose=0,
        )

        validation_probabilities = candidate_model.predict(
            X_validation_dense, verbose=0
        ).ravel()
        validation_predictions = (
            validation_probabilities >= DECISION_THRESHOLD
        ).astype(np.int32)
        run_metrics = binary_classification_metrics(
            y_validation_internal, validation_predictions
        )
        validation_f1_history = np.asarray(history.history["val_f1"])

        optimized_run_records.append(
            {
                "candidate": candidate_name,
                "seed": seed,
                "epochs_run": len(history.history["loss"]),
                "best_epoch": int(validation_f1_history.argmax() + 1),
                "parameter_count": candidate_model.count_params(),
                **run_metrics,
            }
        )

optimized_validation_runs = pd.DataFrame(optimized_run_records)
display(optimized_validation_runs.round(5))
assert optimized_validation_runs.groupby("candidate").size().eq(5).all()
print("Réentraînements optimisés : 10/10")

,candidate,seed,epochs_run,best_epoch,parameter_count,accuracy,precision,recall,f1
0,keras_tuner_optimized_network,11,10,8,1296769,0.87589,0.87591,0.86957,0.87273
1,keras_tuner_optimized_network,22,5,3,1296769,0.87234,0.88636,0.84783,0.86667
2,keras_tuner_optimized_network,33,8,6,1296769,0.88298,0.88889,0.86957,0.87912
3,keras_tuner_optimized_network,44,8,6,1296769,0.87234,0.88060,0.85507,0.86765
4,keras_tuner_optimized_network,55,8,6,1296769,0.86879,0.85816,0.87681,0.86738
5,optuna_optimized_network,11,6,4,648449,0.86879,0.83444,0.91304,0.87197
6,optuna_optimized_network,22,4,2,648449,0.86525,0.86765,0.85507,0.86131
7,optuna_optimized_network,33,9,7,648449,0.84043,0.78182,0.93478,0.85149
8,optuna_optimized_network,44,6,4,648449,0.86170,0.83221,0.89855,0.86411
9,optuna_optimized_network,55,6,4,648449,0.87943,0.83333,0.94203,0.88435


Réentraînements optimisés : 10/10


### 7.3 Application automatique de la règle

La graine d'export sera celle dont le F1 est le plus proche de la moyenne du candidat retenu. Elle représente donc un comportement typique au lieu de sélectionner la meilleure exécution. Le nombre final d'époques sera la médiane des cinq meilleures époques.

In [41]:
optimized_validation_summary = (
    optimized_validation_runs.groupby("candidate")
    .agg(
        runs=("seed", "count"),
        accuracy_mean=("accuracy", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        parameter_count=("parameter_count", "first"),
    )
    .sort_values("f1_mean", ascending=False)
)
display(optimized_validation_summary.round(5))

ranked_by_mean = optimized_validation_summary.sort_values(
    ["f1_mean", "candidate"], ascending=[False, True]
)
mean_gap = float(ranked_by_mean.iloc[0]["f1_mean"] - ranked_by_mean.iloc[1]["f1_mean"])

if mean_gap > PRACTICAL_TIE_TOLERANCE:
    selected_candidate = ranked_by_mean.index[0]
    selection_reason = "mean_f1_gap_above_tolerance"
else:
    tie_rank = optimized_validation_summary.sort_values(
        ["f1_std", "parameter_count", "candidate"],
        ascending=[True, True, True],
    )
    selected_candidate = tie_rank.index[0]
    selection_reason = "practical_tie_resolved_by_stability_then_size"

selected_candidate_mean = float(
    optimized_validation_summary.loc[selected_candidate, "f1_mean"]
)
selected_candidate_runs = optimized_validation_runs[
    optimized_validation_runs["candidate"] == selected_candidate
].copy()
selected_candidate_runs["distance_to_mean"] = (
    selected_candidate_runs["f1"] - selected_candidate_mean
).abs()
representative_run = selected_candidate_runs.sort_values(
    ["distance_to_mean", "seed"], ascending=[True, True]
).iloc[0]
selected_export_seed = int(representative_run["seed"])
selected_epoch_count = int(np.median(selected_candidate_runs["best_epoch"]))

print(f"Architecture retenue : {selected_candidate}")
print(f"Motif                : {selection_reason}")
print(f"Écart des moyennes   : {mean_gap:.5f}")
print(f"Graine représentative: {selected_export_seed}")
print(f"Nombre d'époques fixé: {selected_epoch_count}")

,runs,accuracy_mean,precision_mean,recall_mean,f1_mean,f1_std,parameter_count
candidate,,,,,,,
keras_tuner_optimized_network,5,0.87447,0.87798,0.86377,0.87071,0.00528,1296769
optuna_optimized_network,5,0.86312,0.82989,0.90870,0.86665,0.01232,648449


Architecture retenue : keras_tuner_optimized_network
Motif                : practical_tie_resolved_by_stability_then_size
Écart des moyennes   : 0.00406
Graine représentative: 11
Nombre d'époques fixé: 6


### 7.4 Enregistrement de la décision avant le test

Les résultats et la décision sont sauvegardés avant toute création de matrice du test final. Le prochain commit constituera le point de non-retour méthodologique : après ouverture du test, cette décision ne devra plus être modifiée.

In [42]:
optimized_validation_runs.to_csv(
    RESULTS_DIR / "optimized_network_validation_runs.csv",
    index=False,
    encoding="utf-8",
)
optimized_validation_summary.to_csv(
    RESULTS_DIR / "optimized_network_validation_summary.csv",
    encoding="utf-8",
)

selection_decision = {
    "decision_status": "frozen_before_final_test",
    "final_test_opened": False,
    "selection_rule_file": str(
        MODEL_SELECTION_RULE_PATH.relative_to(PROJECT_ROOT)
    ),
    "candidate_summaries": {
        candidate: {
            key: float(value)
            for key, value in row.items()
        }
        for candidate, row in optimized_validation_summary.to_dict(orient="index").items()
    },
    "mean_f1_gap": mean_gap,
    "selection_reason": selection_reason,
    "selected_candidate": selected_candidate,
    "selected_hyperparameters": optimized_candidates[selected_candidate],
    "selected_export_seed": selected_export_seed,
    "selected_epoch_count": selected_epoch_count,
    "decision_threshold": DECISION_THRESHOLD,
}
with (RESULTS_DIR / "model_selection_decision.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(selection_decision, file, ensure_ascii=False, indent=2)

assert selection_decision["final_test_opened"] is False
assert selected_export_seed in ROBUSTNESS_SEEDS
assert selected_epoch_count >= 1
print("Décision figée avant le test final : OK")

Décision figée avant le test final : OK


### Contrôle de fin de partie 7A

Le résultat attendu est `Réentraînements optimisés : 10/10`, puis `Décision figée avant le test final : OK`. Ne lancez pas encore de transformation ou de métrique sur `test_final`.

Questions de compréhension :

1. Pourquoi choisir la graine la plus proche du F1 moyen est-il plus défendable que choisir la meilleure graine ?
2. Pourquoi faut-il committer cette décision avant de calculer le moindre score sur le test final ?

## Partie 7B — Unique campagne sur le test final

La décision est maintenant figée et commitée. Nous pouvons utiliser les 60 % complets pour ajuster un nouveau TF-IDF et réentraîner le réseau retenu. Le nombre d'époques, la graine, l'architecture, le seuil et tous les hyperparamètres viennent exclusivement de la validation interne.

Le fichier `evidence/metrics/final_test_metrics.json` sert de garde-fou. Il est créé en mode exclusif lors de la campagne : s'il existe déjà, les cellules refusent de recalculer ou d'écraser les métriques du test.

In [43]:
import subprocess
from datetime import datetime

required_names_for_final_test = [
    "train_pool",
    "test_final",
    "production_holdout",
    "TFIDF_PARAMS",
    "optimized_candidates",
    "selected_candidate",
    "selected_export_seed",
    "selected_epoch_count",
    "build_dense_network",
    "binary_classification_metrics",
    "RESULTS_DIR",
]
missing_names = [
    name for name in required_names_for_final_test if name not in globals()
]
assert not missing_names, (
    "Variables manquantes : "
    + ", ".join(missing_names)
    + ". Exécutez les parties précédentes avant la campagne finale."
)

FINAL_TEST_METRICS_PATH = RESULTS_DIR / "final_test_metrics.json"
assert not FINAL_TEST_METRICS_PATH.exists(), (
    f"Campagne finale déjà enregistrée : {FINAL_TEST_METRICS_PATH}. "
    "Ne recalculez pas le test final."
)

with (RESULTS_DIR / "model_selection_decision.json").open(
    "r", encoding="utf-8"
) as file:
    frozen_selection_decision = json.load(file)

assert frozen_selection_decision["final_test_opened"] is False
assert frozen_selection_decision["selected_candidate"] == selected_candidate
assert frozen_selection_decision["selected_export_seed"] == selected_export_seed
assert frozen_selection_decision["selected_epoch_count"] == selected_epoch_count

pre_test_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"Commit avant test : {pre_test_commit}")
print(f"Modèle figé       : {selected_candidate}")
print(f"Graine figée      : {selected_export_seed}")
print(f"Époques figées    : {selected_epoch_count}")
print("Pré-requis de campagne finale : OK")

Commit avant test : fb49780e4b9eb6f5fd91fb42ffabc865603f6f26
Modèle figé       : keras_tuner_optimized_network
Graine figée      : 11
Époques figées    : 6
Pré-requis de campagne finale : OK


### 7.5 Réentraînement sur les 60 % complets

La validation interne a terminé son rôle dès que les choix ont été figés. Ses lignes peuvent donc rejoindre l'apprentissage final. Le vectoriseur est réajusté sur les 1 407 résumés du réservoir d'apprentissage, jamais sur le test. Comme le nombre d'époques est déjà fixé à 6, aucune validation ni early stopping n'est utilisé pendant ce réentraînement.

In [44]:
assert not FINAL_TEST_METRICS_PATH.exists(), "Le test final a déjà été évalué."

final_hyperparameters = optimized_candidates[selected_candidate]
final_tfidf_vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

X_train_final_sparse = final_tfidf_vectorizer.fit_transform(train_pool["resume"])
X_test_final_sparse = final_tfidf_vectorizer.transform(test_final["resume"])
y_train_final = train_pool["cible"].to_numpy(dtype=np.int32)
y_test_final = test_final["cible"].to_numpy(dtype=np.int32)

X_train_final_dense = X_train_final_sparse.toarray()
X_test_final_dense = X_test_final_sparse.toarray()

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(selected_export_seed)
final_model = build_dense_network(
    input_dim=X_train_final_dense.shape[1],
    hidden_units=(final_hyperparameters["units_per_layer"],)
    * final_hyperparameters["num_layers"],
    dropout_rate=final_hyperparameters["dropout_rate"],
    l2_strength=final_hyperparameters["l2_strength"],
    learning_rate=final_hyperparameters["learning_rate"],
)
final_history = final_model.fit(
    X_train_final_dense,
    y_train_final.reshape(-1, 1),
    epochs=selected_epoch_count,
    batch_size=TUNING_BATCH_SIZE,
    verbose=0,
)

print(f"TF-IDF apprentissage final : {X_train_final_dense.shape}")
print(f"TF-IDF test final          : {X_test_final_dense.shape}")
print(f"Paramètres du réseau      : {final_model.count_params():,}")
print(f"Époques réalisées         : {len(final_history.history['loss'])}")
print("Réentraînement final : OK")

TF-IDF apprentissage final : (1407, 10000)
TF-IDF test final          : (469, 10000)
Paramètres du réseau      : 1,296,769
Époques réalisées         : 6
Réentraînement final : OK


### 7.6 Évaluation finale unique

Cette cellule ouvre les résultats du test pour la première et unique fois. La matrice de confusion est organisée `[[vrais négatifs, faux positifs], [faux négatifs, vrais positifs]]`. La probabilité du réseau correspond toujours à la classe `1 = horreur`.

In [45]:
from sklearn.metrics import confusion_matrix

assert not FINAL_TEST_METRICS_PATH.exists(), (
    "Le fichier de métriques finales existe déjà : campagne non relancée."
)

final_test_probabilities = final_model.predict(
    X_test_final_dense, verbose=0
).ravel()
final_test_predictions = (
    final_test_probabilities >= DECISION_THRESHOLD
).astype(np.int32)
final_test_scores = binary_classification_metrics(
    y_test_final, final_test_predictions
)
final_confusion_matrix = confusion_matrix(
    y_test_final, final_test_predictions, labels=[0, 1]
)
final_f1_margin = 1.96 * np.sqrt(
    final_test_scores["f1"]
    * (1 - final_test_scores["f1"])
    / len(y_test_final)
)

final_test_evidence = {
    "campaign_status": "completed_once",
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "pre_test_commit": pre_test_commit,
    "selection_decision_file": "evidence/metrics/model_selection_decision.json",
    "selected_candidate": selected_candidate,
    "selected_hyperparameters": final_hyperparameters,
    "selected_seed": selected_export_seed,
    "epochs": selected_epoch_count,
    "decision_threshold": DECISION_THRESHOLD,
    "train_rows": len(y_train_final),
    "test_rows": len(y_test_final),
    "tfidf_features": X_train_final_dense.shape[1],
    "metrics": {
        name: float(value) for name, value in final_test_scores.items()
    },
    "confusion_matrix_tn_fp_fn_tp": final_confusion_matrix.tolist(),
    "pedagogical_f1_margin": float(final_f1_margin),
    "f1_margin_warning": (
        "Pedagogical approximation requested by the brief; "
        "not an automatically valid confidence interval for F1."
    ),
    "production_holdout_evaluated": False,
    "post_test_retuning_allowed": False,
}

with FINAL_TEST_METRICS_PATH.open("x", encoding="utf-8") as file:
    json.dump(final_test_evidence, file, ensure_ascii=False, indent=2)

display(pd.Series(final_test_scores, name="test_final").to_frame().round(5))
display(
    pd.DataFrame(
        final_confusion_matrix,
        index=["réel_comédie_0", "réel_horreur_1"],
        columns=["prédit_comédie_0", "prédit_horreur_1"],
    )
)
print(f"Marge pédagogique du F1 : ±{final_f1_margin:.4f}")
print("Campagne finale exécutée une seule fois : OK")

,test_final
accuracy,0.84222
precision,0.87864
recall,0.78696
f1,0.83028


,prédit_comédie_0,prédit_horreur_1
réel_comédie_0,214,25
réel_horreur_1,49,181


Marge pédagogique du F1 : ±0.0340
Campagne finale exécutée une seule fois : OK


### Contrôle de fin de partie 7B

Après l'apparition de `Campagne finale exécutée une seule fois : OK`, il est interdit de modifier le TF-IDF, l'architecture, la graine, les époques ou le seuil en fonction du résultat. Un score inférieur aux attentes doit être documenté honnêtement. Le lot `production_holdout` reste inutilisé.

Questions de compréhension :

1. Pourquoi pouvons-nous réintégrer la validation interne dans les 60 % d'apprentissage après avoir figé tous les choix ?
2. Que devons-nous faire si le F1 du test final est moins bon que le F1 de validation ?

## Partie 8 — Sauvegarde et vérification des artefacts

L'objectif est de transformer le résultat encore présent en mémoire en un paquet d'inférence reproductible. Le dossier `model/` contiendra exactement les éléments nécessaires aux prochaines étapes :

- `model.keras` : architecture et poids du réseau, sans état d'entraînement inutile à l'API ;
- `vectorizer.joblib` : vocabulaire, poids IDF et paramètres TF-IDF appris sur les 60 % d'apprentissage ;
- `metrics.json` : identité du modèle, seuil, classes et résultats finaux encodés en UTF-8 ;
- `reference_sample.json` : six textes provenant du lot d'apprentissage et leurs sorties attendues.

L'échantillon de référence prend, sans regarder les prédictions, les trois premières positions de chaque classe dans l'ordre figé du lot d'apprentissage. Il sert uniquement à détecter une différence après rechargement ; ce n'est pas une nouvelle mesure de performance. Le test final n'est donc ni recalculé ni réutilisé et le lot de production reste fermé.

In [ ]:
import joblib

required_names_for_export = [
    "final_model",
    "final_tfidf_vectorizer",
    "final_test_scores",
    "final_test_evidence",
    "train_pool",
    "y_train_final",
    "selected_candidate",
    "final_hyperparameters",
    "selected_export_seed",
    "selected_epoch_count",
    "DECISION_THRESHOLD",
    "FINAL_TEST_METRICS_PATH",
]
missing_names = [
    name for name in required_names_for_export if name not in globals()
]
assert not missing_names, (
    "Variables manquantes : "
    + ", ".join(missing_names)
    + ". Conservez la session ayant réalisé la campagne finale."
)

MODEL_VERSION = "horror-comedy-tfidf-dnn-v1"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_PATH = MODEL_DIR / "model.keras"
VECTORIZER_PATH = MODEL_DIR / "vectorizer.joblib"
METRICS_PATH = MODEL_DIR / "metrics.json"
REFERENCE_SAMPLE_PATH = MODEL_DIR / "reference_sample.json"
ARTIFACT_PATHS = [
    MODEL_PATH,
    VECTORIZER_PATH,
    METRICS_PATH,
    REFERENCE_SAMPLE_PATH,
]

assert FINAL_TEST_METRICS_PATH.exists(), (
    "Les métriques de la campagne finale doivent être enregistrées avant l'export."
)
with FINAL_TEST_METRICS_PATH.open("r", encoding="utf-8") as file:
    recorded_final_metrics = json.load(file)

assert recorded_final_metrics["campaign_status"] == "completed_once"
assert recorded_final_metrics["production_holdout_evaluated"] is False
assert recorded_final_metrics["post_test_retuning_allowed"] is False
assert np.isclose(
    recorded_final_metrics["metrics"]["f1"], final_test_scores["f1"]
)

existing_artifacts = [path for path in ARTIFACT_PATHS if path.exists()]
assert not existing_artifacts, (
    "Artefacts déjà présents : "
    + ", ".join(str(path) for path in existing_artifacts)
    + ". Ne les écrasez pas sans diagnostic."
)

export_source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"Version du modèle : {MODEL_VERSION}")
print(f"Commit source     : {export_source_commit}")
print(f"Dossier cible     : {MODEL_DIR}")
print("Pré-requis de sauvegarde : OK")

### 8.1 Création du paquet d'inférence

Le clone sauvegardé reprend exactement l'architecture et les poids du réseau final, mais pas l'optimiseur : l'API fera uniquement de l'inférence. Le vectoriseur doit impérativement voyager avec le réseau, car les colonnes de la matrice doivent correspondre au même vocabulaire et aux mêmes poids IDF.

La version du modèle (`horror-comedy-tfidf-dnn-v1`) identifie ce paquet appris. Elle sera distincte de la version de l'application, qui correspondra plus tard au commit Git déployé.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# from_config reconstruit l'architecture sans recopier la compilation ni Adam.
inference_model = tf.keras.Sequential.from_config(final_model.get_config())
inference_model.set_weights(final_model.get_weights())
assert getattr(inference_model, "optimizer", None) is None
inference_model.save(MODEL_PATH)
joblib.dump(final_tfidf_vectorizer, VECTORIZER_PATH)

portable_metrics = {
    "schema_version": 1,
    "model_version": MODEL_VERSION,
    "task": "classification_binaire_resume_film",
    "model_file": MODEL_PATH.name,
    "vectorizer_file": VECTORIZER_PATH.name,
    "reference_sample_file": REFERENCE_SAMPLE_PATH.name,
    "label_mapping": {"0": "comédie", "1": "horreur"},
    "probability_class": 1,
    "decision_threshold": float(DECISION_THRESHOLD),
    "selected_candidate": selected_candidate,
    "selected_hyperparameters": final_hyperparameters,
    "selected_seed": int(selected_export_seed),
    "epochs": int(selected_epoch_count),
    "train_rows": int(recorded_final_metrics["train_rows"]),
    "test_rows": int(recorded_final_metrics["test_rows"]),
    "tfidf_features": int(recorded_final_metrics["tfidf_features"]),
    "metrics": recorded_final_metrics["metrics"],
    "confusion_matrix_tn_fp_fn_tp": recorded_final_metrics[
        "confusion_matrix_tn_fp_fn_tp"
    ],
    "pedagogical_f1_margin": recorded_final_metrics[
        "pedagogical_f1_margin"
    ],
    "f1_margin_warning": recorded_final_metrics["f1_margin_warning"],
    "production_holdout_evaluated": False,
    "source_commit": export_source_commit,
    "pre_test_commit": recorded_final_metrics["pre_test_commit"],
    "exported_at": datetime.now().astimezone().isoformat(timespec="seconds"),
}
with METRICS_PATH.open("x", encoding="utf-8") as file:
    json.dump(portable_metrics, file, ensure_ascii=False, indent=2)
    file.write("\n")

# Sélection indépendante des sorties : trois premières positions de chaque classe.
reference_positions = np.concatenate(
    [np.flatnonzero(y_train_final == label)[:3] for label in (0, 1)]
)
reference_rows = train_pool.iloc[reference_positions].reset_index(drop=True)
X_reference = final_tfidf_vectorizer.transform(reference_rows["resume"]).toarray()
reference_probabilities = final_model.predict(X_reference, verbose=0).ravel()
reference_predictions = (
    reference_probabilities >= DECISION_THRESHOLD
).astype(np.int32)

reference_records = []
for row_index, row in reference_rows.iterrows():
    reference_records.append(
        {
            "clean_row_id": int(row["clean_row_id"]),
            "titre": str(row["titre"]),
            "resume": str(row["resume"]),
            "true_label": int(row["cible"]),
            "expected_probability_class_1": float(
                reference_probabilities[row_index]
            ),
            "expected_class": int(reference_predictions[row_index]),
        }
    )

reference_payload = {
    "schema_version": 1,
    "model_version": MODEL_VERSION,
    "provenance": {
        "source": "train_pool (60 % d'apprentissage après dédoublonnage)",
        "selection_rule": (
            "trois premières positions de chaque classe dans l'ordre figé "
            "du split, sans regarder les prédictions"
        ),
        "purpose": "test technique de reproductibilité après rechargement",
        "not_for_performance_evaluation": True,
    },
    "probability_class": 1,
    "decision_threshold": float(DECISION_THRESHOLD),
    "samples": reference_records,
}
with REFERENCE_SAMPLE_PATH.open("x", encoding="utf-8") as file:
    json.dump(reference_payload, file, ensure_ascii=False, indent=2)
    file.write("\n")

assert all(path.exists() for path in ARTIFACT_PATHS)
print("Artefacts sauvegardés : 4/4")
for path in ARTIFACT_PATHS:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

### 8.2 Rechargement indépendant

Nous rechargeons les fichiers depuis le disque, sans réutiliser les objets Python sauvegardés en mémoire pour produire les nouvelles sorties. Les classes prédites doivent être identiques et les probabilités égales à une petite tolérance près (`rtol=10⁻⁶`, `atol=10⁻⁷`) afin de tenir compte des minuscules variations numériques possibles.

In [ ]:
reloaded_vectorizer = joblib.load(VECTORIZER_PATH)
reloaded_model = tf.keras.models.load_model(MODEL_PATH, compile=False)

with METRICS_PATH.open("r", encoding="utf-8") as file:
    reloaded_metrics = json.load(file)
with REFERENCE_SAMPLE_PATH.open("r", encoding="utf-8") as file:
    reloaded_reference = json.load(file)

reference_texts = [item["resume"] for item in reloaded_reference["samples"]]
expected_probabilities = np.array(
    [
        item["expected_probability_class_1"]
        for item in reloaded_reference["samples"]
    ],
    dtype=np.float32,
)
expected_classes = np.array(
    [item["expected_class"] for item in reloaded_reference["samples"]],
    dtype=np.int32,
)

X_reference_reloaded = reloaded_vectorizer.transform(reference_texts).toarray()
reloaded_probabilities = reloaded_model.predict(
    X_reference_reloaded, verbose=0
).ravel()
reloaded_classes = (
    reloaded_probabilities >= reloaded_metrics["decision_threshold"]
).astype(np.int32)

np.testing.assert_allclose(
    reloaded_probabilities,
    expected_probabilities,
    rtol=1e-6,
    atol=1e-7,
)
np.testing.assert_array_equal(reloaded_classes, expected_classes)
assert X_reference_reloaded.shape == (6, reloaded_metrics["tfidf_features"])
assert reloaded_metrics["metrics"] == recorded_final_metrics["metrics"]
assert reloaded_reference["provenance"]["not_for_performance_evaluation"]
assert "comédie" in METRICS_PATH.read_text(encoding="utf-8")

artifact_inventory = pd.DataFrame(
    [
        {
            "fichier": str(path.relative_to(PROJECT_ROOT)),
            "taille_Mio": path.stat().st_size / (1024**2),
        }
        for path in ARTIFACT_PATHS
    ]
)
display(artifact_inventory.round({"taille_Mio": 3}))
max_probability_delta = float(
    np.max(np.abs(reloaded_probabilities - expected_probabilities))
)
print(f"Échantillons vérifiés       : {len(reference_texts)}")
print(f"Écart maximal de probabilité: {max_probability_delta:.10f}")
print("Rechargement et reproduction des prédictions : OK")

### Contrôle de fin de partie 8

Le résultat attendu est `Artefacts sauvegardés : 4/4`, puis `Rechargement et reproduction des prédictions : OK`. La taille exacte des fichiers et l'écart numérique seront des résultats réellement observés au moment de l'exécution.

Questions de compréhension :

1. Pourquoi le réseau seul ne suffit-il pas pour prédire correctement à partir d'un texte brut ?
2. Pourquoi l'échantillon de référence vérifie-t-il la reproductibilité technique, mais pas la performance de généralisation ?